# 07 | Contribution, repeat buying and pricing risk

**Author: Chanakya**

Separate marginal acquisition economics from fixed-rights viability. All unobserved costs and behavioural persistence are explicit scenarios. No revenue coverage is called profit.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/manifests/release.json').exists())
sys.path.insert(0, str(ROOT))
from src.analysis_common import *
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
rng = np.random.default_rng(CFG['seed'])
print('Offline inputs:', CFG['raw_release'], '| Author: Chanakya')
import src.analysis_common as shared
shared.ACTIVE_NOTEBOOK='07_contribution_and_pricing'
shared.ACTIVE_SOURCES=['fancode_terms', 'v3_cloudflare_stream_pricing', 'v3_fancode_terms', 'v3_razorpay_pricing', 'v5_gst_council_rates']

Offline inputs: raw-v5-2026-09-14 | Author: Chanakya


## 1. Contribution bridge
For tax-inclusive price P: net revenue = P/(1+tax). Gateway charge uses an illustrative external 2% list fee plus 18% fee tax, conservatively expensed with no input-tax recovery. Other variable cost is additional to that fee. Rights cost is excluded from marginal contribution and assessed separately. Tax-exclusive prices are a separate ledger: tax charged to customers is not revenue.

In [2]:
def contribution(price,tax=.18,variable=30,fee=.02,fee_tax=.18):
 return price/(1+tax)-price*fee*(1+fee_tax)-variable
rows=[]
for price in [39,49,79,89,99,116,199,399,899,999]:
 for tax in CFG['gst_scenarios']:
  for variable in CFG['variable_cost_scenarios']:
   c=contribution(price,tax,variable)
   for cac in CFG['acquisition_cost_scenarios']:rows.append(dict(price=price,tax=tax,other_variable_cost=variable,cac=cac,net_sales=price/(1+tax),gateway_cost=price*.02*1.18,pre_marketing_contribution=c,after_acquisition=c-cac,contribution_pct_receipts=c/price,scope='Assumption scenario, not observed FanCode margin'))
econ=pd.DataFrame(rows);table(econ,'07_contribution_grid');table(econ,'contribution_grid',True)
base=econ[(econ.tax==.18)&(econ.other_variable_cost==30)&(econ.cac==175)];display(base)
plt.figure(figsize=(10,4));plt.bar(base.price.astype(str),base.pre_marketing_contribution,label='Before acquisition');plt.axhline(175,color=COLORS[3],linestyle='--',label='INR175 acquisition scenario');plt.ylabel('INR per purchase');plt.xlabel('Customer price (INR)');plt.title('Short passes cannot bear the same acquisition cost as seasons');plt.legend();fig('07_contribution_bridge','18% tax-inclusive scenario, 2% gateway + fee tax, INR30 other variable cost. Rights excluded.')
# Explicit tax-exclusive alternative changes customer payable as well as revenue.
exclusive=pd.DataFrame([dict(list_before_tax=p,payable=p*1.18,net_sales=p,contribution=p-p*1.18*.02*1.18-30) for p in [89,399,999]])
display(table(exclusive,'07_tax_exclusive_alternative'))

,price,tax,other_variable_cost,cac,net_sales,gateway_cost,pre_marketing_contribution,after_acquisition,contribution_pct_receipts,scope
18,39,0.180,30,175,33.051,0.920,2.130,-172.870,0.055,"Assumption scenario, not observed FanCode margin"
42,49,0.180,30,175,41.525,1.156,10.369,-164.631,0.212,"Assumption scenario, not observed FanCode margin"
66,79,0.180,30,175,66.949,1.864,35.085,-139.915,0.444,"Assumption scenario, not observed FanCode margin"
90,89,0.180,30,175,75.424,2.100,43.323,-131.677,0.487,"Assumption scenario, not observed FanCode margin"
114,99,0.180,30,175,83.898,2.336,51.562,-123.438,0.521,"Assumption scenario, not observed FanCode margin"
138,116,0.180,30,175,98.305,2.738,65.567,-109.433,0.565,"Assumption scenario, not observed FanCode margin"
162,199,0.180,30,175,168.644,4.696,133.948,-41.052,0.673,"Assumption scenario, not observed FanCode margin"
186,399,0.180,30,175,338.136,9.416,298.719,123.719,0.749,"Assumption scenario, not observed FanCode margin"
210,899,0.180,30,175,761.864,21.216,710.648,535.648,0.790,"Assumption scenario, not observed FanCode margin"
234,999,0.180,30,175,846.610,23.576,793.034,618.034,0.794,"Assumption scenario, not observed FanCode margin"


<Figure size 1000x400 with 1 Axes>

,list_before_tax,payable,net_sales,contribution
0,89,105.020,89,56.522
1,399,470.820,399,357.889
2,999,"1,178.820",999,941.180


## 2. Repeat purchase, with horizon definitions kept separate
The supplied 60–65% is only a repurchase proportion. First show exactly one possible repeat. Then separately test a stationary geometric process capped at K purchase opportunities. E[purchases] = sum(p^j, j=0..K-1). This is a sensitivity model, not an estimated lifetime.

In [3]:
repeat=[]
for p in [.60,.65]:
 for k in [2,3,5,10]:
  purchases=sum(p**j for j in range(k))
  for v in [10,30,60]:
   c=contribution(89,variable=v);repeat.append(dict(repurchase_probability=p,opportunity_cap=k,expected_purchases=purchases,net_revenue=89/1.18*purchases,contribution_before_acquisition=c*purchases,contribution_after_175=c*purchases-175,variable=v))
repeat=pd.DataFrame(repeat);display(table(repeat,'07_repeat_horizons'))
plt.figure(figsize=(9,4))
for p,g in repeat[repeat.variable==30].groupby('repurchase_probability'):plt.plot(g.opportunity_cap,g.contribution_after_175,marker='o',label=f'p={p:.0%}')
plt.axhline(0,color='black',lw=.8);plt.xlabel('Maximum purchase opportunities');plt.ylabel('Expected contribution after INR175 CAC');plt.title('Repeat purchases help only if per-order contribution survives');plt.legend();fig('07_repeat_sensitivity','INR89 midpoint tournament scenario. Constant repeat probability is unverified. No infinite lifetime claim.')
ltv=[]
for start in [2026,2027,2028]:
 for r in CFG['renewal_scenarios']:
  for d in CFG['discount_rate_scenarios']:
   years=2028-start+1;c=contribution(399);pv=sum(c*r**j/(1+d)**j for j in range(years));ltv.append(dict(acquisition_year=start,renewal=r,discount_rate=d,covered_seasons=years,pre_marketing_pv=pv,after_175=pv-175))
ltv=pd.DataFrame(ltv);display(table(ltv,'07_rights_horizon_ltv'))
print('Full-season INR399 receipts even for late-2026 acquisition would be optimistic. Use actual remaining-season receipts before launch. Post-2028 value is excluded, not asserted to be zero for the whole company.')

,repurchase_probability,opportunity_cap,expected_purchases,net_revenue,contribution_before_acquisition,contribution_after_175,variable
0,0.600,2,1.600,120.678,101.317,-73.683,10
1,0.600,2,1.600,120.678,69.317,-105.683,30
2,0.600,2,1.600,120.678,21.317,-153.683,60
3,0.600,3,1.960,147.831,124.114,-50.886,10
4,0.600,3,1.960,147.831,84.914,-90.086,30
5,0.600,3,1.960,147.831,26.114,-148.886,60
6,0.600,5,2.306,173.897,145.998,-29.002,10
7,0.600,5,2.306,173.897,99.886,-75.114,30
8,0.600,5,2.306,173.897,30.718,-144.282,60
9,0.600,10,2.485,187.419,157.351,-17.649,10


<Figure size 900x400 with 1 Axes>

,acquisition_year,renewal,discount_rate,covered_seasons,pre_marketing_pv,after_175
0,2026,0.700,0.000,3,654.195,479.195
1,2026,0.700,0.100,3,609.782,434.782
2,2026,0.700,0.200,3,574.620,399.620
3,2026,0.800,0.000,3,728.875,553.875
4,2026,0.800,0.100,3,673.970,498.970
...,...,...,...,...,...,...
22,2028,0.800,0.100,1,298.719,123.719
23,2028,0.800,0.200,1,298.719,123.719
24,2028,0.900,0.000,1,298.719,123.719
25,2028,0.900,0.100,1,298.719,123.719


Full-season INR399 receipts even for late-2026 acquisition would be optimistic. Use actual remaining-season receipts before launch. Post-2028 value is excluded, not asserted to be zero for the whole company.


## 3. Credit leakage and discount hurdle
For qualifying tournament price E and season price S, control upgrade probability u0 and treatment u1, entry contribution cancels. Delta = u1*C(S-credit) - u0*C(S). Break-even u1/u0 = C(S)/C(S-credit). This assumes the same entry cohort and one upgrade maximum. Full randomized portfolio contribution is the operational primary outcome.

In [4]:
leak=[]
for credit in [0,39,44.5,50,89]:
 for u0 in [.05,.10,.20,.40]:
  c0=contribution(399);c1=contribution(399-credit);threshold=u0*c0/c1
  leak.append(dict(credit=credit,control_upgrade=u0,required_treatment_upgrade=threshold,required_relative_lift=c0/c1-1,leakage_per_organic_upgrade=c0-c1,qualifier='Same entry cohort and costs'))
leak=pd.DataFrame(leak);display(table(leak,'07_credit_break_even'))
discounts=[]
for price in [39,79,89,99,399]:
 for v in [0,10,30,60]:
  old=contribution(price,variable=v);new=contribution(price*.9,variable=v);discounts.append(dict(price=price,variable=v,old_contribution=old,new_contribution=new,required_volume_uplift=old/new-1 if old>0 and new>0 else np.nan))
discounts=pd.DataFrame(discounts);table(discounts,'07_discount_hurdles')
p=discounts.pivot(index='variable',columns='price',values='required_volume_uplift');plt.figure(figsize=(9,3.5));plt.imshow(p,aspect='auto',cmap='YlOrRd');plt.xticks(range(len(p.columns)),p.columns);plt.yticks(range(len(p)),p.index);plt.colorbar(label='Required relative volume uplift');plt.xlabel('Original price (INR)');plt.ylabel('Other variable cost (INR)');plt.title('A 10% price cut can need much more than 11.1% extra volume');fig('07_discount_hurdles','NaN means the positive-contribution hurdle is not defined. No elasticity measured.')

,credit,control_upgrade,required_treatment_upgrade,required_relative_lift,leakage_per_organic_upgrade,qualifier
0,0.000,0.050,0.050,0.000,0.000,Same entry cohort and costs
1,0.000,0.100,0.100,0.000,0.000,Same entry cohort and costs
2,0.000,0.200,0.200,0.000,0.000,Same entry cohort and costs
3,0.000,0.400,0.400,0.000,0.000,Same entry cohort and costs
4,39.000,0.050,0.056,0.121,32.130,Same entry cohort and costs
5,39.000,0.100,0.112,0.121,32.130,Same entry cohort and costs
6,39.000,0.200,0.224,0.121,32.130,Same entry cohort and costs
7,39.000,0.400,0.448,0.121,32.130,Same entry cohort and costs
8,44.500,0.050,0.057,0.140,36.662,Same entry cohort and costs
9,44.500,0.100,0.114,0.140,36.662,Same entry cohort and costs


<Figure size 900x350 with 2 Axes>

## 4. Usage and reward costs can reverse the result
The Cloudflare public delivery rate is an external stress benchmark, not FanCode procurement evidence. INR80/90/100 per USD are FX scenarios, not current exchange-rate claims. Do not add this delivery estimate to a variable-cost scenario that already includes delivery. Referral voucher face value is not necessarily cash cost.

In [5]:
usage=[]
for price in [89,399,999]:
 for minutes in [90,600,1800,3600]:
  for fx in [80,90,100]:
   delivery=minutes/1000*float(METRICS['stream_delivery']['value'])*fx
   net=price/1.18-price*.02*1.18-10-delivery
   usage.append(dict(price=price,watch_minutes=minutes,usd_inr_assumption=fx,delivery_benchmark_cost=delivery,other_operations_assumption=10,pre_marketing_contribution=net,after_175=net-175))
usage=pd.DataFrame(usage);display(table(usage,'07_usage_cost_stress'))
plt.figure(figsize=(9,4))
for price,g in usage[usage.usd_inr_assumption==90].groupby('price'):plt.plot(g.watch_minutes,g.after_175,marker='o',label=f'INR{price} receipt')
plt.axhline(0,color='black',lw=.8);plt.xlabel('Delivered watch minutes per buyer over the evaluated term');plt.ylabel('Contribution after INR175 acquisition');plt.title('More viewing also carries a delivery-cost obligation');plt.legend();fig('07_usage_cost_stress','External USD1/1,000-minute benchmark, assumed FX90, INR10 operations. No FanCode contract inference.')
rewards=[]
for price,face in [(89,150),(399,500)]:
 for redemption in [.25,.5,1]:
  for procurement_fraction in [.1,.5,1]:
   reward_cost=face*redemption*procurement_fraction;rewards.append(dict(price=price,voucher_face=face,redemption_assumption=redemption,procurement_fraction_assumption=procurement_fraction,expected_reward_cost=reward_cost,contribution_after_reward=contribution(price)-reward_cost,source_id='v3_fancode_terms'))
display(table(pd.DataFrame(rewards),'07_referral_reward_stress'))
print('Reward funding, redemption and procurement terms are unknown. Published voucher face value must not be described as CAC or cash expense.')

,price,watch_minutes,usd_inr_assumption,delivery_benchmark_cost,other_operations_assumption,pre_marketing_contribution,after_175
0,89,90,80,7.200,10,56.123,-118.877
1,89,90,90,8.100,10,55.223,-119.777
2,89,90,100,9.000,10,54.323,-120.677
3,89,600,80,48.000,10,15.323,-159.677
4,89,600,90,54.000,10,9.323,-165.677
...,...,...,...,...,...,...,...
31,999,1800,90,162.000,10,651.034,476.034
32,999,1800,100,180.000,10,633.034,458.034
33,999,3600,80,288.000,10,525.034,350.034
34,999,3600,90,324.000,10,489.034,314.034


<Figure size 900x400 with 1 Axes>

,price,voucher_face,redemption_assumption,procurement_fraction_assumption,expected_reward_cost,contribution_after_reward,source_id
0,89,150,0.250,0.100,3.750,39.573,v3_fancode_terms
1,89,150,0.250,0.500,18.750,24.573,v3_fancode_terms
2,89,150,0.250,1.000,37.500,5.823,v3_fancode_terms
3,89,150,0.500,0.100,7.500,35.823,v3_fancode_terms
4,89,150,0.500,0.500,37.500,5.823,v3_fancode_terms
5,89,150,0.500,1.000,75.000,-31.677,v3_fancode_terms
6,89,150,1.000,0.100,15.000,28.323,v3_fancode_terms
7,89,150,1.000,0.500,75.000,-31.677,v3_fancode_terms
8,89,150,1.000,1.000,150.000,-106.677,v3_fancode_terms
9,399,500,0.250,0.100,12.500,286.219,v3_fancode_terms


Reward funding, redemption and procurement terms are unknown. Published voucher face value must not be described as CAC or cash expense.


## 5. Fixed-cost coverage and true acquisition
The supplied INR150–200 target concerns acquired paying subscribers. The brief does not define a causal incremental denominator. Applying INR200 to incremental CAC is a proposed additional hurdle. Attributed INR175 below is illustrative, not achieved performance. A normalized INR1 crore fixed budget is not the ATP rights fee. Required payers = fixed cost / positive contribution after acquisition. If the denominator is zero or negative, no finite payer scale covers the fixed cost in that scenario.

In [6]:
fixed=[]
for c in [0,150,175,200]:
 for v in [10,30,60]:
  net=contribution(399,variable=v)-c;fixed.append(dict(fixed_cost=1e7,cac=c,variable=v,contribution_per_payer=net,required_new_payers=np.ceil(1e7/net) if net>0 else np.nan))
display(table(pd.DataFrame(fixed),'07_fixed_cost_coverage'))
inc=pd.DataFrame([dict(reported_cac=c,incremental_fraction=q,true_icac=c/q) for c in [150,175,200] for q in np.arange(.1,1.01,.05)]);table(inc,'07_incrementality_cac')
plt.figure(figsize=(9,4))
for c,g in inc.groupby('reported_cac'):plt.plot(g.incremental_fraction,g.true_icac,label=f'Attributed INR{c}')
plt.axhline(200,color='black',ls='--');plt.ylim(0,1200);plt.xlabel('Fraction of credited payers who are incremental new payers');plt.ylabel('Incremental CAC (INR)');plt.title('Attribution efficiency is not acquisition efficiency');plt.legend();fig('07_incrementality_frontier','Attributed INR175 is hypothetical. 87.5% incrementality is required for a proposed INR200 incremental-CAC hurdle, not a supplied causal target.')
check('07_economics',{'tax_bridge':abs(399/1.18*1.18-399)<1e-8,'credit_no_effect_at_zero':bool((leak[leak.credit==0].required_relative_lift==0).all()),'discount_proportional_hurdle':abs((1/.9-1)-.1111111111)<1e-8,'incremental_fraction_hurdle':175/200==.875,'cost_grid_identity':bool(np.allclose(econ.pre_marketing_contribution-econ.cac,econ.after_acquisition)),'finite_repeat_below_infinite':all(r.expected_purchases<1/(1-r.repurchase_probability) for r in repeat.itertuples())})
report('07_economics_findings',f'Base scenario season contribution before acquisition is INR{contribution(399):.2f}, after INR175 acquisition is INR{contribution(399)-175:.2f}. These are scenario outputs, not company margins. Attributed INR175 requires 87.5% incremental new payers to meet INR200 iCAC. Capped credits need additional upgrades to offset leakage. Rights costs remain a normalized coverage frontier.')

,fixed_cost,cac,variable,contribution_per_payer,required_new_payers
0,"10,000,000.000",0,10,318.719,"31,376.000"
1,"10,000,000.000",0,30,298.719,"33,477.000"
2,"10,000,000.000",0,60,268.719,"37,214.000"
3,"10,000,000.000",150,10,168.719,"59,271.000"
4,"10,000,000.000",150,30,148.719,"67,241.000"
5,"10,000,000.000",150,60,118.719,"84,233.000"
6,"10,000,000.000",175,10,143.719,"69,581.000"
7,"10,000,000.000",175,30,123.719,"80,829.000"
8,"10,000,000.000",175,60,93.719,"106,702.000"
9,"10,000,000.000",200,10,118.719,"84,233.000"


<Figure size 900x400 with 1 Axes>

,check,passed
0,tax_bridge,True
1,credit_no_effect_at_zero,True
2,discount_proportional_hurdle,True
3,incremental_fraction_hurdle,True
4,cost_grid_identity,True
5,finite_repeat_below_infinite,True


## Source references
These IDs resolve to the preserved bodies, URLs and capture timestamps. Derived tables also retain row-level source IDs where applicable. Case inputs refer to the supplied brief, physical PDF pages 9–14. Review source files resolve through the review collection log. Scenario parameters are in analysis_config.

In [7]:
references=source_table(['fancode_terms', 'v3_cloudflare_stream_pricing', 'v3_fancode_terms', 'v3_razorpay_pricing', 'v5_gst_council_rates'])
display(table(references,'07_source_references'))

,source_id,url,raw_file,retrieved
0,fancode_terms,https://www.fancode.com/about/tnc,data/raw/offers/fancode_terms__ec711b2c0c24.html,2026-09-13T15:31:30.110677+00:00
1,v3_cloudflare_stream_pricing,https://developers.cloudflare.com/stream/pricing/,data/raw/cost_benchmarks/v3_cloudflare_stream_...,2026-09-14T12:13:56.708882+00:00
2,v3_fancode_terms,https://www.fancode.com/about/tnc,data/raw/pricing/v3_fancode_terms__f8543d68e21...,2026-09-14T12:17:35.629941+00:00
3,v3_razorpay_pricing,https://razorpay.com/pricing/,data/raw/cost_benchmarks/v3_razorpay_pricing__...,2026-09-14T12:13:56.696422+00:00
4,v5_gst_council_rates,https://www.gstcouncil.gov.in/sites/default/fi...,data/raw/pricing/v5_gst_council_rates__f1d4414...,2026-09-14T17:01:48.432769+00:00
